import os, warnings, torch

import scanpy as sc
import pandas as pd

from datasets.data_manager_human_lymph_node import Lymph_node
from utils.notebook_graph_utils import build_graph_model, collect_predictions, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


In [ ]:
import os, warnings, torch

import scanpy as sc
import pandas as pd

from datasets.data_manager_human_lymph_node import Lymph_node
from utils.notebook_graph_utils import build_graph_model, collect_predictions, load_graph_checkpoint
from utils.utils import *
from utils.utils_dataloader import *
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from palettable.cartocolors.diverging import *
from palettable.scientific.diverging import *


### Load dataset

In [ ]:
rna_path = '/home/wzk/ST_data/2024_nmethods_SpatialGlue_Human_lymph_node_3slides/slice2/s2_adata_rna.h5ad'
protein_path = '/home/wzk/ST_data/2024_nmethods_SpatialGlue_Human_lymph_node_3slides/slice2/s2_adata_adt.h5ad'

adata_rna_testing = sc.read_h5ad(rna_path)
adata_protein_testing = sc.read_h5ad(protein_path)

### Load args

In [ ]:
%run ./args/args_human_lymph_node.py
args = args

### Create dataloader

In [ ]:
# create the dataset
dataset = Lymph_node(
    adata_path=args.adata_path,
    n_top_genes=args.n_source,
    graph_k=args.graph_k,
    val_ratio=args.val_ratio,
    mask_seed=args.seed,
)


### Model initialization

In [ ]:
# create the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_graph_model(args, dataset, use_cell_type=False).to(device)
model = load_graph_checkpoint(model, 'WholeSliceGraphTransformer_human_lymph_node_last.pth', device=device)


### Model inference 

In [ ]:
pd_value, gt_value, _ = collect_predictions(
    model,
    dataset.testing,
    split='test',
    device=device,
)

pd_value = pd_value.numpy()
gt_value = gt_value.numpy()


### Model evaluation

In [ ]:
from utils.evaluation import evaluator
pearson_sample_list, spearman_sample_list, _ = evaluator(pd_value, gt_value)

print(pearson_sample_list.mean())
print(spearman_sample_list.mean())

In [ ]:
pd_value = np.exp((pd_value * dataset.std) + dataset.mean)
gt_value = np.exp((gt_value * dataset.std) + dataset.mean)

In [ ]:
proteins = adata_protein_testing.var['gene_ids'].values

for index, protein in enumerate(proteins):
    adata_protein_testing.obs['pd_' + protein ] = pd_value[:, index]
    adata_protein_testing.obs['gt_' + protein ] = gt_value[:, index]
    

In [ ]:
protein = 'HLA-DRA'

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
sc.pl.embedding(adata_protein_testing, basis='spatial', color='gt_' + protein, title=f'Ground Truth {protein}', ax=axs[0], show=False, cmap=Tropic_7.mpl_colormap, size=30, vmax=90000) 
sc.pl.embedding(adata_protein_testing, basis='spatial', color='pd_' + protein, title=f'Prediction {protein}', ax=axs[1], show=False, cmap=Tropic_7.mpl_colormap, size=30) 

In [ ]:
protein = 'PAX5'

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
sc.pl.embedding(adata_protein_testing, basis='spatial', color='gt_' + protein, title=f'Ground Truth {protein}', ax=axs[0], show=False, cmap=Tropic_7.mpl_colormap, size=30) 
sc.pl.embedding(adata_protein_testing, basis='spatial', color='pd_' + protein, title=f'Prediction {protein}', ax=axs[1], show=False, cmap=Tropic_7.mpl_colormap, size=30) 